<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [5]</a>'.</span>

# CPE Level Complete — Ineligible Combo Prediction Bias

Analyzes **`target_model_bias_pct`** and **hypothetical product bias** for
auction requests that are **gated out** by the eligibility gate in `v11_cpe_lc_v2` /
`v11_cpe_lc_v3`.

**Gating signal (in `mz_dcpi_prediction_v1`):**
```
body.app_event_p > 0   -- model has a non-zero prediction
AND body.cst = 0       -- gate zeroed the bid cost
AND body.max_cst > 0   -- active campaign with a target CPE
```

**Risk of removing the gate:**  
- High `gated_model_bias_pct` → model overestimates conversion → overspend risk  
- Low bias similar to eligible cohort → model is calibrated → safe to unlock  

**Sections:**
1. Overall: gated vs eligible model bias + hypothetical product bias  
2. Platform: iOS vs Android for each cohort  
3. Prediction percentile: calibration curve for gated vs eligible  
4. Game × Event breakdown for gated combos (main risk table)  
5. Risk ranking: ungating candidates sorted by volume × |bias|  

> **Note on product bias for gated rows:** Since `cost = 0` for gated auctions,
> classical product bias is trivially −100%. Instead we compute
> **hypothetical product bias** = `SUM(target_cpe × pred) / SUM(actual_cpe) − 1`,
> which equals model bias (assuming no discount factor). This represents the
> financial calibration *if the gate were removed*.

In [1]:
import warnings
warnings.filterwarnings('ignore')

from google.cloud import bigquery
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

client = bigquery.Client(project='unity-ads-ds-prd')

ANALYSIS_START = '2026-06-15'
# Exclude known downtime
DOWNTIME_START = '2026-07-25'
DOWNTIME_END   = '2026-08-06'

print('BigQuery client initialized')
print(f'Analysis window: {ANALYSIS_START} → today (excl {DOWNTIME_START}–{DOWNTIME_END})')

BigQuery client initialized
Analysis window: 2026-06-15 → today (excl 2026-07-25–2026-08-06)


In [2]:
# ---------------------------------------------------------------------------
# Shared BigQuery CTEs
# preds: adds is_gated flag to split cohorts
# camps: active LEVEL_COMPLETE campaigns
# outs:  D7 outcome data
# ---------------------------------------------------------------------------
CTES = f'''
WITH preds AS (
  SELECT
    p.body.auction_id  AS auction_id,
    p.body.app_event_p AS pred,
    p.body.cst         AS cost,
    p.body.max_cst     AS target_cpe,
    p.body.campaign_id AS campaign_id,
    (p.body.cst = 0)   AS is_gated,
    DATE_DIFF(CURRENT_DATE(), p.submit_date, DAY) AS age_days
  FROM `unity-ai-data-prd.mz_dcpi_raw.mz_dcpi_prediction_v1` AS p
  WHERE p.submit_date >= \'{ANALYSIS_START}\'
    AND NOT (p.submit_date BETWEEN \'{DOWNTIME_START}\' AND \'{DOWNTIME_END}\')
    AND p.body.app_event_p > 0
    AND p.body.app_event_type = \'level_complete\'
    AND p.body.max_cst > 0
),
camps AS (
  SELECT
    id AS campaign_id,
    game_id AS target_game_id,
    IFNULL(sdk_event_names, []) AS target_events,
    ARRAY_LENGTH(IFNULL(sdk_event_names, [])) = 0 AS is_wildcard
  FROM `unity-data-ads-core-prd.ads_dimension_data.campaigns_v3`
  WHERE app_event_conversion_type = \'LEVEL_COMPLETE\'
    AND archived_at IS NULL
),
outs AS (
  SELECT
    o.auctionId,
    o.app_event_level_complete_count_d7,
    o.app_event_level_complete_sdk_event_name_array,
    LOWER(o.platform) AS platform,
    o.country
  FROM `unity-data-ads-core-prd.ads_secondary_conversion.operativeecpm_installs_outcomes_contextual` AS o
  WHERE DATE(o.adRequestTimestamp) >= \'{ANALYSIS_START}\'
    AND NOT (DATE(o.adRequestTimestamp) BETWEEN \'{DOWNTIME_START}\' AND \'{DOWNTIME_END}\')
    AND o.campaignType = \'appEventConversion\'
)'''

# ---------------------------------------------------------------------------
# Event-match subquery: checks if any fired LC event matches campaign target
# ---------------------------------------------------------------------------
_EV_MATCH = '''(SELECT COUNT(1)
               FROM UNNEST(outs.app_event_level_complete_sdk_event_name_array.list) AS ev,
                    UNNEST(c.target_events) AS tgt_ev
               WHERE LOWER(ev.element) = LOWER(tgt_ev)) > 0'''

# ---------------------------------------------------------------------------
# Bias SELECT expressions
# Model bias  = SUM(pred)             / SUM(actual_conversion) - 1
# Product bias = SUM(cost)             / SUM(actual_cpe)        - 1  (eligible only)
# Hypo bias   = SUM(target_cpe * pred) / SUM(actual_cpe)        - 1  (gated only)
# ---------------------------------------------------------------------------
def _model_bias_expr(cohort_filter='', alias='model_bias_pct'):
    cf = f'AND {cohort_filter}' if cohort_filter else ''
    return f'''
  ROUND(100 * (
    SUM(CASE WHEN age_days > 8 AND c.campaign_id IS NOT NULL {cf} THEN p.pred END)
    / NULLIF(
        SUM(CASE WHEN age_days > 8 AND c.campaign_id IS NOT NULL {cf}
                 THEN CASE
                        WHEN outs.app_event_level_complete_count_d7 = 0 THEN 0.0
                        WHEN c.is_wildcard THEN 1.0
                        ELSE IF({_EV_MATCH}, 1.0, 0.0)
                      END
            END), 0
      ) - 1), 2) AS {alias}'''

def _product_bias_expr(cohort_filter='', alias='product_bias_pct', use_hypo=False):
    cf = f'AND {cohort_filter}' if cohort_filter else ''
    # For gated rows use target_cpe * pred as hypothetical cost
    cost_expr = 'p.target_cpe * p.pred' if use_hypo else 'p.cost'
    return f'''
  ROUND(100 * (
    SUM(CASE WHEN age_days >= 9 AND c.campaign_id IS NOT NULL {cf} THEN {cost_expr} END)
    / NULLIF(
        SUM(CASE WHEN age_days >= 9 AND c.campaign_id IS NOT NULL {cf}
                 THEN CASE
                        WHEN outs.app_event_level_complete_count_d7 = 0 THEN 0.0
                        WHEN c.is_wildcard THEN p.target_cpe
                        ELSE IF({_EV_MATCH}, p.target_cpe, 0.0)
                      END
            END), 0
      ) - 1), 2) AS {alias}'''

JOINS = '''FROM preds AS p
INNER JOIN outs ON outs.auctionId = p.auction_id
LEFT JOIN camps AS c ON c.campaign_id = p.campaign_id'''

def run_query(sql):
    return client.query(sql).to_dataframe()

print('CTEs and helpers defined.')

CTEs and helpers defined.


In [3]:
# ---------------------------------------------------------------------------
# Plotting helpers
# ---------------------------------------------------------------------------
COLORS = {
    'gated_model':    '#e8541e',   # orange-red
    'gated_product':  '#c83000',   # dark red
    'eligible_model': '#7b4fa8',   # purple
    'eligible_product': '#e8a020', # amber
}

def plot_cohort_bars(df, dim_col, title, top_n=None, sort_by='gated_auction_count',
                    min_gated=50, height_per_row=30):
    """Horizontal grouped bar showing model bias for gated vs eligible cohorts."""
    df = df.copy()
    if min_gated > 0:
        df = df[df.get('gated_auction_count', pd.Series([999]*len(df))) >= min_gated]
    if top_n and len(df) > top_n:
        df = df.nlargest(top_n, sort_by)
    df = df.sort_values(sort_by, ascending=True).reset_index(drop=True)
    y_labels = df[dim_col].astype(str)

    fig = go.Figure()
    for col, label, color in [
        ('gated_model_bias_pct',    'Gated — model bias',    COLORS['gated_model']),
        ('eligible_model_bias_pct', 'Eligible — model bias', COLORS['eligible_model']),
    ]:
        if col in df.columns:
            fig.add_trace(go.Bar(
                y=y_labels, x=df[col], name=label, orientation='h',
                marker_color=color, opacity=0.85,
                hovertemplate=f'%{{y}}<br>{label}: %{{x:.1f}}%<extra></extra>',
            ))

    fig.add_vline(x=20,  line_dash='dot', line_color='lightgray', annotation_text='+20%')
    fig.add_vline(x=-20, line_dash='dot', line_color='lightgray', annotation_text='-20%')
    fig.add_vline(x=0,   line_color='black', line_width=1)

    fig.update_layout(
        title=title, barmode='group',
        height=max(450, len(df) * height_per_row + 180),
        xaxis_title='Model Bias %', template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    )
    fig.show()

print('Plot helpers defined.')

Plot helpers defined.


---
## 1. Overall: Gated vs Eligible Cohort Bias

Aggregate model bias and hypothetical product bias for each cohort over the full analysis window.
This sets the baseline before drilling into platform / prediction-level / combo dimensions.

In [4]:
sql_overall = CTES + f''',
ranked AS (
  SELECT *
  FROM preds
)
SELECT
  is_gated,
  COUNT(*)              AS auction_count,
  ROUND(SUM(CASE WHEN NOT is_gated THEN p.cost ELSE 0 END) / 1e6, 2) AS total_spend_usd,
  ROUND(AVG(p.pred) * 100, 4)  AS avg_pred_pct,
  {_model_bias_expr(alias='model_bias_pct')},
  {_product_bias_expr(use_hypo=False, alias='eligible_product_bias_pct')},
  {_product_bias_expr(use_hypo=True,  alias='hypo_product_bias_pct')}
{JOINS}
GROUP BY is_gated
ORDER BY is_gated
'''

df_overall = run_query(sql_overall)
df_overall['cohort'] = df_overall['is_gated'].map({True: 'Gated (ineligible)', False: 'Eligible'})
display(df_overall[['cohort','auction_count','total_spend_usd','avg_pred_pct',
                     'model_bias_pct','eligible_product_bias_pct','hypo_product_bias_pct']])

,cohort,auction_count,total_spend_usd,avg_pred_pct,model_bias_pct,eligible_product_bias_pct,hypo_product_bias_pct
0,Eligible,2324013,4136507.3,24.5634,19.27,51.19,51.19


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [5]:
gated_row   = df_overall[df_overall['is_gated'] == True].iloc[0]
eligible_row = df_overall[df_overall['is_gated'] == False].iloc[0]

fig = go.Figure()
categories = ['Model Bias', 'Product Bias (eligible cost)', 'Hypo Product Bias (gated)']
gated_vals   = [gated_row['model_bias_pct'],   None, gated_row['hypo_product_bias_pct']]
eligible_vals = [eligible_row['model_bias_pct'], eligible_row['eligible_product_bias_pct'], None]

fig.add_trace(go.Bar(name='Gated',   x=categories, y=gated_vals,   marker_color=COLORS['gated_model']))
fig.add_trace(go.Bar(name='Eligible', x=categories, y=eligible_vals, marker_color=COLORS['eligible_model']))
fig.add_hline(y=0,   line_color='black', line_width=1)
fig.add_hline(y=20,  line_dash='dot', line_color='lightgray', annotation_text='+20%')
fig.add_hline(y=-20, line_dash='dot', line_color='lightgray', annotation_text='-20%')

fig.update_layout(
    title='Overall Bias: Gated vs Eligible Cohorts (since 2026-06-15, age ≥ 9d)',
    yaxis_title='Bias %', barmode='group', height=450, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

gated_mb   = gated_row['model_bias_pct']
eligible_mb = eligible_row['model_bias_pct']
print(f'Gated model bias:   {gated_mb:+.1f}%')
print(f'Eligible model bias:{eligible_mb:+.1f}%')
print(f'Difference:         {gated_mb - eligible_mb:+.1f}pp')

IndexError: single positional indexer is out-of-bounds

---
## 2. Platform — iOS vs Android

Per-platform model bias for gated vs eligible.  
A large divergence between platforms within the gated cohort indicates  
platform-specific calibration risks for gate removal.

In [ ]:
sql_platform = CTES + f'''
SELECT
  outs.platform,
  COUNTIF(p.is_gated)                           AS gated_auction_count,
  COUNTIF(NOT p.is_gated)                       AS eligible_auction_count,
  {_model_bias_expr('p.is_gated',     'gated_model_bias_pct')},
  {_model_bias_expr('NOT p.is_gated', 'eligible_model_bias_pct')},
  {_product_bias_expr('NOT p.is_gated', use_hypo=False, alias='eligible_product_bias_pct')},
  {_product_bias_expr('p.is_gated',   use_hypo=True,  alias='gated_hypo_product_bias_pct')}
{JOINS}
GROUP BY outs.platform
ORDER BY outs.platform
'''

df_platform = run_query(sql_platform)
display(df_platform)

In [ ]:
platforms = df_platform['platform'].tolist()

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Gated — model bias',
    x=platforms, y=df_platform['gated_model_bias_pct'],
    marker_color=COLORS['gated_model'], opacity=0.85,
))
fig.add_trace(go.Bar(
    name='Eligible — model bias',
    x=platforms, y=df_platform['eligible_model_bias_pct'],
    marker_color=COLORS['eligible_model'], opacity=0.85,
))
fig.add_trace(go.Bar(
    name='Eligible — product bias',
    x=platforms, y=df_platform['eligible_product_bias_pct'],
    marker_color=COLORS['eligible_product'], opacity=0.75,
))

fig.add_hline(y=0,   line_color='black', line_width=1)
fig.add_hline(y=20,  line_dash='dot', line_color='lightgray', annotation_text='+20%')
fig.add_hline(y=-20, line_dash='dot', line_color='lightgray', annotation_text='-20%')

fig.update_layout(
    title='Bias by Platform — Gated vs Eligible (since 2026-06-15, age ≥ 9d)',
    yaxis_title='Bias %', barmode='group', height=450, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

---
## 3. Prediction Percentile — Calibration Curve

Predictions ranked into **20 equal-sized buckets** via `NTILE(20)`.
Comparing gated vs eligible bias across the confidence spectrum reveals
whether miscalibration is concentrated at specific prediction ranges.

- If gated bias is consistently higher → systematic overestimation of unseen combos  
- If gated bias tracks eligible bias → model generalises well to unseen combos

In [ ]:
sql_pctile = CTES + ''',
ranked AS (
  SELECT *, NTILE(20) OVER (ORDER BY pred) AS pred_bucket
  FROM preds
)''' + f'''

SELECT
  pred_bucket,
  ROUND(MIN(p.pred), 6)        AS pred_min,
  ROUND(MAX(p.pred), 6)        AS pred_max,
  COUNTIF(p.is_gated)          AS gated_count,
  COUNTIF(NOT p.is_gated)      AS eligible_count,
  ROUND(AVG(IF(p.is_gated,     p.pred, NULL)) * 100, 4) AS gated_avg_pred_pct,
  ROUND(AVG(IF(NOT p.is_gated, p.pred, NULL)) * 100, 4) AS eligible_avg_pred_pct,
  {_model_bias_expr('p.is_gated',     'gated_model_bias_pct')},
  {_model_bias_expr('NOT p.is_gated', 'eligible_model_bias_pct')}
FROM ranked AS p
INNER JOIN outs ON outs.auctionId = p.auction_id
LEFT JOIN camps AS c ON c.campaign_id = p.campaign_id
GROUP BY pred_bucket
ORDER BY pred_bucket
'''

df_pctile = run_query(sql_pctile)
display(df_pctile)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_pctile['pred_bucket'], y=df_pctile['gated_model_bias_pct'],
    name='Gated — model bias', mode='lines+markers',
    line=dict(color=COLORS['gated_model']),
))
fig.add_trace(go.Scatter(
    x=df_pctile['pred_bucket'], y=df_pctile['eligible_model_bias_pct'],
    name='Eligible — model bias', mode='lines+markers',
    line=dict(color=COLORS['eligible_model']),
))

fig.add_hline(y=20,  line_dash='dot', line_color='lightgray', annotation_text='+20%')
fig.add_hline(y=-20, line_dash='dot', line_color='lightgray', annotation_text='-20%')
fig.add_hline(y=0,   line_color='black', line_width=1)

fig.update_layout(
    title='Model Bias by Prediction Percentile NTILE(20) — Gated vs Eligible',
    xaxis_title='Prediction Bucket (1=lowest CVR → 20=highest)',
    yaxis_title='Model Bias %',
    template='plotly_white', height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

---
## 4. Game × Event Breakdown — Gated Combos

Per `(target_game_id, target_event)` pair: model bias, actual LC rate, and
auction volumes for the **gated** cohort only.

Pairs with < 100 gated auctions are excluded for statistical reliability.

In [ ]:
sql_game_ev = CTES + ''',
camp_ev AS (
  SELECT c.campaign_id, c.target_game_id, LOWER(ev) AS target_event
  FROM camps AS c, UNNEST(c.target_events) AS ev
  WHERE NOT c.is_wildcard
),
matched AS (
  SELECT
    p.pred, p.cost, p.target_cpe, p.age_days, p.is_gated,
    ce.target_game_id, ce.target_event,
    IF(
      (SELECT COUNT(1)
       FROM UNNEST(outs.app_event_level_complete_sdk_event_name_array.list) AS ev_item
       WHERE LOWER(ev_item.element) = ce.target_event) > 0,
      1.0, 0.0
    ) AS ev_matched
  FROM preds AS p
  INNER JOIN outs ON outs.auctionId = p.auction_id
  INNER JOIN camp_ev AS ce ON ce.campaign_id = p.campaign_id
  WHERE p.is_gated
)
SELECT
  target_game_id,
  target_event,
  COUNT(*)                                                            AS gated_auction_count,
  ROUND(AVG(pred) * 100, 4)                                           AS avg_pred_pct,
  ROUND(SAFE_DIVIDE(SUM(ev_matched), COUNT(*)) * 100, 4)              AS actual_lcr_pct,
  ROUND(100 * SAFE_DIVIDE(
    SUM(IF(age_days > 8,  pred,       NULL)),
    SUM(IF(age_days > 8,  ev_matched, NULL))
  ) - 100, 2)                                                         AS gated_model_bias_pct,
  ROUND(100 * SAFE_DIVIDE(
    SUM(IF(age_days >= 9, target_cpe * pred, NULL)),
    SUM(IF(age_days >= 9, ev_matched * target_cpe, NULL))
  ) - 100, 2)                                                         AS gated_hypo_product_bias_pct,
  COUNTIF(ev_matched > 0 AND age_days > 8)                           AS observed_conversions_d8
FROM matched
GROUP BY target_game_id, target_event
HAVING COUNT(*) >= 100
ORDER BY gated_auction_count DESC
'''

df_game_ev = run_query(sql_game_ev)
print(f'{len(df_game_ev)} (game_id, event) pairs with >= 100 gated auctions')
display(df_game_ev)

In [ ]:
# Scatter: actual_lcr_pct vs gated_model_bias_pct
# Size = gated_auction_count, color = bias magnitude
df_s = df_game_ev.dropna(subset=['gated_model_bias_pct', 'actual_lcr_pct']).copy()
df_s['label'] = df_s['target_game_id'].astype(str) + ' / ' + df_s['target_event']

fig = px.scatter(
    df_s,
    x='actual_lcr_pct',
    y='gated_model_bias_pct',
    size='gated_auction_count',
    color='gated_model_bias_pct',
    color_continuous_scale='RdYlGn_r',
    color_continuous_midpoint=20,
    hover_name='label',
    hover_data=['gated_auction_count', 'avg_pred_pct', 'actual_lcr_pct', 'observed_conversions_d8'],
    title='Gated Combos: Actual LC Rate vs Model Bias — size = gated auction count',
    labels={'actual_lcr_pct': 'Actual LC Rate (%)', 'gated_model_bias_pct': 'Gated Model Bias (%)'},
    template='plotly_white',
    height=600,
)
fig.add_hline(y=0,   line_color='black', line_width=1)
fig.add_hline(y=20,  line_dash='dot', line_color='gray', annotation_text='+20%')
fig.add_hline(y=-20, line_dash='dot', line_color='gray', annotation_text='-20%')
fig.show()

In [ ]:
# Top 30 by gated volume — horizontal bar
top30 = df_game_ev.dropna(subset=['gated_model_bias_pct']).nlargest(30, 'gated_auction_count').copy()
top30 = top30.sort_values('gated_auction_count', ascending=True)
top30['combo'] = top30['target_game_id'].astype(str) + ' / ' + top30['target_event']

fig = go.Figure()
fig.add_trace(go.Bar(
    y=top30['combo'], x=top30['gated_model_bias_pct'],
    name='Gated model bias', orientation='h',
    marker_color=COLORS['gated_model'], opacity=0.85,
    customdata=top30[['gated_auction_count', 'actual_lcr_pct', 'avg_pred_pct']].values,
    hovertemplate=('%{y}<br>model bias: %{x:.1f}%<br>'
                   'auctions: %{customdata[0]:,}<br>'
                   'actual_lcr: %{customdata[1]:.3f}%<br>'
                   'avg_pred: %{customdata[2]:.3f}%<extra></extra>'),
))
fig.add_vline(x=20,  line_dash='dot', line_color='lightgray', annotation_text='+20%')
fig.add_vline(x=-20, line_dash='dot', line_color='lightgray', annotation_text='-20%')
fig.add_vline(x=0,   line_color='black', line_width=1)

fig.update_layout(
    title='Gated Model Bias — Top 30 Combos by Gated Volume',
    xaxis_title='Model Bias %', template='plotly_white',
    height=max(500, len(top30) * 25 + 200),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

---
## 5. Risk Ranking — Ungating Candidates

Scoring each gated `(game_id, event)` combo for gate-removal risk:

```
risk_score = log10(gated_auction_count) × |gated_model_bias_pct|
```

- **Low risk** (recommended to unlock): high volume + bias near zero  
- **High risk** (keep gated): high volume + large positive bias (model overestimates)  
- **Negative bias combos**: model underestimates → unlocking may be safe or unprofitable  

Combos where `observed_conversions_d8 < 5` are flagged as data-sparse.

In [ ]:
import numpy as np

df_risk = df_game_ev.dropna(subset=['gated_model_bias_pct']).copy()
df_risk['log_volume'] = np.log10(df_risk['gated_auction_count'].clip(lower=1))
df_risk['abs_bias']   = df_risk['gated_model_bias_pct'].abs()
df_risk['risk_score'] = df_risk['log_volume'] * df_risk['abs_bias']
df_risk['data_sparse'] = df_risk['observed_conversions_d8'] < 5
df_risk['combo'] = df_risk['target_game_id'].astype(str) + ' / ' + df_risk['target_event']

# Low-risk candidates: high volume, |bias| < 20%, reasonable data
low_risk = df_risk[
    (df_risk['abs_bias'] < 20) &
    (df_risk['gated_auction_count'] >= 500) &
    (~df_risk['data_sparse'])
].sort_values('gated_auction_count', ascending=False)

# High-risk combos: large positive bias (model overestimates)
high_risk = df_risk[
    (df_risk['gated_model_bias_pct'] > 50) &
    (df_risk['gated_auction_count'] >= 500) &
    (~df_risk['data_sparse'])
].sort_values('risk_score', ascending=False)

print(f'=== LOW RISK (|bias| < 20%, volume >= 500, sufficient data): {len(low_risk)} combos ===')
display(low_risk[['combo','gated_auction_count','avg_pred_pct','actual_lcr_pct',
                   'gated_model_bias_pct','observed_conversions_d8']].head(20))

print(f'\n=== HIGH RISK (model_bias > 50%, volume >= 500): {len(high_risk)} combos ===')
display(high_risk[['combo','gated_auction_count','avg_pred_pct','actual_lcr_pct',
                    'gated_model_bias_pct','risk_score']].head(20))

In [ ]:
# Full risk ranking scatter: volume vs bias, colored by risk tier
df_r = df_risk.copy()
df_r['risk_tier'] = 'Medium'
df_r.loc[df_r['abs_bias'] < 20, 'risk_tier'] = 'Low'
df_r.loc[df_r['gated_model_bias_pct'] > 50, 'risk_tier'] = 'High'
df_r.loc[df_r['data_sparse'], 'risk_tier'] = 'Data sparse'

color_map = {'Low': '#2ca02c', 'Medium': '#ff7f0e', 'High': '#d62728', 'Data sparse': '#aec7e8'}

fig = px.scatter(
    df_r,
    x='gated_auction_count',
    y='gated_model_bias_pct',
    color='risk_tier',
    color_discrete_map=color_map,
    size='abs_bias',
    hover_name='combo',
    hover_data=['gated_auction_count', 'actual_lcr_pct', 'avg_pred_pct', 'observed_conversions_d8'],
    log_x=True,
    title='Gate Removal Risk: Volume vs Model Bias (log scale x)',
    labels={
        'gated_auction_count': 'Gated Auctions (log)',
        'gated_model_bias_pct': 'Gated Model Bias (%)',
    },
    template='plotly_white',
    height=600,
)
fig.add_hline(y=20,  line_dash='dot', line_color='orange', annotation_text='+20% threshold')
fig.add_hline(y=50,  line_dash='dot', line_color='red',    annotation_text='+50% high risk')
fig.add_hline(y=-20, line_dash='dot', line_color='gray',   annotation_text='-20%')
fig.add_hline(y=0,   line_color='black', line_width=1)
fig.show()

---
## 6. Summary — Gate Removal Risk Assessment

In [ ]:
total_combos   = len(df_risk)
low_risk_n     = len(df_risk[(df_risk['abs_bias'] < 20)  & (~df_risk['data_sparse'])])
medium_risk_n  = len(df_risk[(df_risk['abs_bias'].between(20, 50)) & (~df_risk['data_sparse'])])
high_risk_n    = len(df_risk[(df_risk['gated_model_bias_pct'] > 50) & (~df_risk['data_sparse'])])
sparse_n       = int(df_risk['data_sparse'].sum())

gated_mb_str   = f"{gated_row['model_bias_pct']:+.1f}%"
eligible_mb_str = f"{eligible_row['model_bias_pct']:+.1f}%"

print('=' * 65)
print('GATE REMOVAL RISK SUMMARY')
print('=' * 65)
print(f'Analysis window : {ANALYSIS_START} → today (excl downtime)')
print(f'Gated combos    : {total_combos} (game_id, event) pairs with >= 100 auctions')
print()
print('OVERALL BIAS')
print(f'  Gated model bias   : {gated_mb_str}')
print(f'  Eligible model bias: {eligible_mb_str}')
print()
print('COMBO-LEVEL RISK BREAKDOWN')
print(f'  Low risk  (|bias| < 20%, data ok) : {low_risk_n:3d} combos  → candidates to unlock')
print(f'  Medium    (20% ≤ |bias| ≤ 50%)    : {medium_risk_n:3d} combos  → monitor post-unlock')
print(f'  High risk (bias > 50%)            : {high_risk_n:3d} combos  → keep gated or retrain')
print(f'  Data sparse (< 5 conversions)     : {sparse_n:3d} combos  → insufficient data')
print()
print('INTERPRETATION')
if abs(gated_row['model_bias_pct'] - eligible_row['model_bias_pct']) < 20:
    print('  Gated bias is SIMILAR to eligible → model generalises to unseen combos.')
    print('  Gate removal is LOWER risk overall; review per-combo table before unlocking.')
else:
    delta = gated_row['model_bias_pct'] - eligible_row['model_bias_pct']
    print(f'  Gated bias is {delta:+.1f}pp vs eligible → systematic overestimation on unseen combos.')
    print('  Gate removal carries HIGHER risk; prioritise low-risk combos only.')

print('=' * 65)